<a href="https://colab.research.google.com/github/khu55/PINN-for-Chem/blob/main/PINN_for_chem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# ── 1. 活化能数据（从论文提取） ──────────────────────────
Ea = {
    'k1': 289.53e3,  # TBBT → INT14
    'k2': 108.72e3,  # INT14 → INT21
    'k3': 170.91e3,  # INT21 → INT31
    'k4': 84.44e3,   # INT31 → INT41
    'k5': 171.21e3,  # INT41 → INT51
}

R = 8.314  # J/(mol·K)

# ── 2. 估算pre-exponential factor A ──────────────────────
# 依据：DSC观测到154.6°C分解，假设TBBT半衰期约5分钟
T_decomp = 154.6 + 273.15
k1_target = np.log(2) / 300       # 半衰期300秒 → k1 ≈ 2.3e-3 s⁻¹
A = k1_target / np.exp(-Ea['k1'] / (R * T_decomp))
print(f"Estimated A: {A:.4e}")


In [ ]:
# ── 3. 速率常数函数 ───────────────────────────────────────
def get_k(T_celsius):
    T = T_celsius + 273.15
    return {name: A * np.exp(-ea / (R * T))
            for name, ea in Ea.items()}

k = get_k(154.6)
print("\n速率常数 at 154.6°C:")
for name, val in k.items():
    print(f"  {name}: {val:.4e} s⁻¹")


In [ ]:
# ── 4. ODE系统 ────────────────────────────────────────────
def ode_system(t, y, k):
    TBBT, INT14, INT21, INT31, INT41, INT51 = y
    dTBBT  = -k['k1'] * TBBT
    dINT14 =  k['k1'] * TBBT  - k['k2'] * INT14
    dINT21 =  k['k2'] * INT14 - k['k3'] * INT21
    dINT31 =  k['k3'] * INT21 - k['k4'] * INT31
    dINT41 =  k['k4'] * INT31 - k['k5'] * INT41
    dINT51 =  k['k5'] * INT41
    return [dTBBT, dINT14, dINT21, dINT31, dINT41, dINT51]


In [ ]:
# ── 5. 求解ODE ────────────────────────────────────────────
y0 = [1.0, 0, 0, 0, 0, 0]  # 初始浓度归一化
t_span = [0, 600]           # 0~600秒（10分钟）

sol = solve_ivp(ode_system, t_span, y0, args=(k,),
                dense_output=True, max_step=1,
                method='Radau')

In [ ]:
# ── 6. 画图 ───────────────────────────────────────────────
t = np.linspace(0, 600, 1000)
y = sol.sol(t)
labels = ['TBBT', 'INT14', 'INT21', 'INT31', 'INT41', 'INT51']

plt.figure(figsize=(10, 5))
for i, label in enumerate(labels):
    plt.plot(t, y[i], label=label)
plt.xlabel('Time (s)')
plt.ylabel('Concentration (normalized)')
plt.title('TBBT Decomposition at 154.6°C')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# ── 1. 参数 ───────────────────────────────────────────────
T_MAX = 600.0
k1_val = np.log(2) / 300        # 只用k1
k1_scaled = k1_val * T_MAX      # 归一化后的k1 ≈ 1.386，数值安全

print(f"k1_scaled: {k1_scaled:.4f}")  # 应该是1.386

# ── 2. 数据（从sol取） ────────────────────────────────────
t_np = np.linspace(0, 1, 200)
y_np = sol.sol(t_np * T_MAX).T   # (200, 6)

t_tensor = torch.tensor(t_np, dtype=torch.float32).unsqueeze(1)
y_tensor = torch.tensor(y_np,  dtype=torch.float32)

# ── 3. 网络 ───────────────────────────────────────────────
class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 6),  nn.Sigmoid()
        )
    def forward(self, t):
        return self.net(t)

torch.manual_seed(42)
model = PINN()

# ── 4. 物理损失（只用k1）─────────────────────────────────
def physics_loss(model, t_phys):
    t_phys = t_phys.clone().requires_grad_(True)
    u = model(t_phys)
    g = torch.autograd.grad(
        u[:, 0].sum(), t_phys, create_graph=True
    )[0].squeeze()
    r0 = g + k1_scaled * u[:, 0]
    return (r0 ** 2).mean()

# ── 5. 训练 ───────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
t_phys = torch.linspace(0, 1, 500).unsqueeze(1)

losses, losses_data, losses_phys = [], [], []

for epoch in range(5000):
    optimizer.zero_grad()

    u_pred  = model(t_tensor)
    l_data  = nn.MSELoss()(u_pred, y_tensor)
    l_phys  = physics_loss(model, t_phys)
    loss    = l_data + 0.1 * l_phys

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())
    losses_data.append(l_data.item())
    losses_phys.append(l_phys.item())

    if epoch % 500 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss.item():.6f} | "
              f"Data: {l_data.item():.6f} | Phys: {l_phys.item():.6f}")

# ── 6. Loss曲线 ───────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(losses,       label='Total')
plt.plot(losses_data,  label='Data')
plt.plot(losses_phys,  label='Physics')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ── 7. 预测对比图 ─────────────────────────────────────────
model.eval()
with torch.no_grad():
    t_test = torch.linspace(0, 1, 1000).unsqueeze(1)
    y_pred = model(t_test).numpy()

t_plot = np.linspace(0, 600, 1000)
y_ref  = sol.sol(t_plot)  # solve_ivp参考值
labels = ['TBBT', 'INT14', 'INT21', 'INT31', 'INT41', 'INT51']

plt.figure(figsize=(10, 5))
for i, label in enumerate(labels):
    plt.plot(t_plot, y_ref[i],    '--', alpha=0.6, label=f'{label} (ODE)')
    plt.plot(t_plot, y_pred[:, i], '-',             label=f'{label} (PINN)')

plt.xlabel('Time (s)')
plt.ylabel('Concentration (normalized)')
plt.title('PINN vs ODE Reference')
plt.legend(ncol=2, fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 只给前300秒的数据（前半段），预测后半段
idx_half = np.where(t_np <= 0.5)[0]  # 只取t<=0.5（即300秒内）
t_half = t_tensor[idx_half]
y_half = y_tensor[idx_half]

print(f"训练数据点数: {len(idx_half)}")  # 应该是100个点

# 重新训练普通NN
torch.manual_seed(42)
model_plain = PINN()
optimizer_plain = torch.optim.Adam(model_plain.parameters(), lr=1e-3)

for epoch in range(5000):
    optimizer_plain.zero_grad()
    loss = nn.MSELoss()(model_plain(t_half), y_half)
    loss.backward()
    optimizer_plain.step()
    if epoch % 1000 == 0:
        print(f"Plain | Epoch {epoch} | Loss: {loss.item():.6f}")

# 重新训练PINN
torch.manual_seed(42)
model_pinn2 = PINN()
optimizer_pinn2 = torch.optim.Adam(model_pinn2.parameters(), lr=1e-3)

for epoch in range(5000):
    optimizer_pinn2.zero_grad()
    l_data = nn.MSELoss()(model_pinn2(t_half), y_half)
    l_phys = physics_loss(model_pinn2, t_phys)
    loss = l_data + 0.1 * l_phys
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_pinn2.parameters(), 1.0)
    optimizer_pinn2.step()
    if epoch % 1000 == 0:
        print(f"PINN  | Epoch {epoch} | Loss: {loss.item():.6f}")

# 对比图
model_plain.eval()
model_pinn2.eval()
with torch.no_grad():
    t_test = torch.linspace(0, 1, 1000).unsqueeze(1)
    y_plain2 = model_plain(t_test).numpy()
    y_pinn2  = model_pinn2(t_test).numpy()

t_plot = np.linspace(0, 600, 1000)
y_ref  = sol.sol(t_plot)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, i, name in zip(axes, [0, 5], ['TBBT', 'INT51']):
    ax.plot(t_plot, y_ref[i],     'k--', linewidth=2, label='ODE真实值')
    ax.plot(t_plot, y_plain2[:, i], 'r-', label='普通NN')
    ax.plot(t_plot, y_pinn2[:, i],  'b-', label='PINN')
    ax.axvline(x=300, color='gray', linestyle=':', label='数据截止点')
    ax.set_title(name)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Concentration')
    ax.legend()
    ax.grid(True)

plt.suptitle('外推能力对比：只用前300秒数据，预测后300秒', fontsize=13)
plt.tight_layout()
plt.show()